# Totimorphic cell

This notebook constructs a totimorphic cell made from bars and joints using the [bar_joint_model class](../api.html#quivira.bar_joint_model).

In [1]:
# Project and symbolic tools
import heyoka as hy
import quivira as qv

# Numerics and visualization
import numpy as np
from _animation import animate_bar_model

## Building the cell

The model of the cell is built using point pairs representing the start and endpoints of bars. The joints are found from overlaps using [`infer_joints`](../api.html#quivira.bar_joint_model.infer_joints).

A torque is added with [`add_external_force`](../api.html#quivira.bar_joint_model.add_external_force). This function can either take constant values that are baked into the equations of motion, or heyoka parameters whose values can be adjusted after the equations of motion are constructed.

In [2]:
model = qv.bar_joint_model()

# Define bars as pairs of endpoints
point_pairs = [
    ((0, 0), (1, 4)),
    ((1, 4), (0, 8)),
    ((0, 8), (4, 7)),
    ((4, 7), (8, 8)),
    ((8, 8), (9, 4)),
    ((9, 4), (8, 0)),
    ((8, 0), (4, -1)),
    ((4, -1), (0, 0)),
]

# Add all bars to the model
for start, end in point_pairs:
    model.add_bar_from_point_pair(start, end)

model.infer_joints()

model.add_support(bar_index=0, s=-0.5, dof_in_x=False, dof_in_y=False)

model.add_external_force(bar_index=3, s=0, fx=0, fy=0, torque=hy.par[0])

ode_system = model.build_lagrange_eom()

# Taylor integrator

The torque is set as heyoka parameter during integration

In [3]:
torque = -40.0

ta = hy.taylor_adaptive(ode_system, pars=[torque], compact_mode=True)

state0 = model.get_initial_state()
model.check_constraint_fulfillment(state0)

## Simulation

In [4]:
ta.time = 0.0
ta.state[:] = state0
t_end = 1.0
tgrid = np.linspace(0.0, t_end, 201)
solution = ta.propagate_grid(tgrid)[-1]


Animate

In [5]:
animate_bar_model(model, solution, torque_values=[torque])